#### 1. Основные функции в координации e-commerce

Аналитик обеспечивает связность процессов, предоставляя единую версию правды (Single Source of Truth) для всех отделов:

*   **Мониторинг KPI в реальном времени:** Аналитик создает и поддерживает дашборды (например, в Apache Superset), отслеживая ключевые показатели: выручка, конверсия (CR), 
средний чек (AOV), стоимость привлечения клиента (CAC), пожизненная ценность (LTV).

*   **Координация маркетинга и продаж:** Аналитик выявляет, какие каналы приносят самую высокую прибыль, и рекомендуют перераспределять бюджет. Он анализирует эффективность акций и промокодов, координируя действия отдела продаж с маркетингом.

*   **Оптимизация воронки продаж (CRO):** Аналитик находит узкие места в пути пользователя (где происходит отказ от корзины, где долгая загрузка) и координирует работу с дизайнерами и разработчиками для улучшения UX/UI.

*   **Синхронизация с логистикой и складом:** Аналитик прогнозирует спрос на основе исторических данных, помогая избежать дефицита (out-of-stock) или затоваривания.

#### 2. Практические сценарии (Co-working)

**Маркетинг** Анализ ROAS (возврат расходов на рекламу), когортный анализ, эффективность e-mail рассылок.

**Product/UX** A/B тестирование новых фич сайта, анализ поведения пользователей (тепловые карты, пути).

**Категорийный менеджмент** ABC/XYZ-анализ ассортимента, поиск товаров с высокой маржой, анализ ценообразования конкурентов.

**Логистика** Анализ скорости доставки, точности комплектации заказов (order accuracy).

#### 3. Ценность для бизнеса

Аналитик данных в роли координатора позволяет:

*   **Принимать обоснованные решения:** Переход от интуитивного управления к управлению на основе данных.

*   **Быстро реагировать:** Мгновенно  замечать аномалии (например, резкое падение конверсии) и координировать их устранение.

*   **Персонализировать опыт:** Сегментировать клиентов для повышения повторных продаж.

В итоге, аналитик данных становится **"мозговым центром"**, который координирует все части e-commerce механизма для максимизации прибыли.

### Работа аналитика в интернет-магазине электронной техники

Жизненный цикл гаджетов короткий, а цена ошибки (затоваривание склада устаревшими моделями или дефицит новинок) крайне высока. 

#### 1. Сбор и подготовка данных (PostgreSQL)

Первым шагом аналитик извлекает исторические данные о продажах, остатках и поступлениях. В электронике важно учитывать не только количество, но и категории (напримеп, "Смартфоны" продаются быстрее, чем "Холодильники").
**Пример  SQL-запроса для выгрузки агрегатных данных:**

In [ ]:
# Настройка путей и импортов
import sys
from pathlib import Path

# Найдём корень проекта (где лежит README.md или config.py)
def find_project_root(marker="config.py"):
    current = Path().resolve()
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent
    raise RuntimeError(f"Project root with '{marker}' not found!")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

# Добавим корень проекта в sys.path, чтобы импортировать src и config
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Путь к данным
CSV_PATH = PROJECT_ROOT / "data" / "raw" / "report.csv"
assert CSV_PATH.exists(), f"CSV file not found at {CSV_PATH}"
print(f"CSV file: {CSV_PATH}")

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from config import RAW_DATA_DIR
from src.db.queries import run_query


In [8]:
# Автоматически перезагружает модули при изменении кода в .py-файлах.
# Не нужно перезапускать ядро после каждого изменения в src/.

# %load_ext autoreload
# %autoreload 2

In [4]:
df = pd.read_csv(
    CSV_PATH,                      # путь к файлу
    sep=';',                       # разделитель - точка с запятой
    parse_dates=['sale_date'],     # колонка для преобразования в дату
    encoding='utf-8',              # кодировка файла
    dtype={'product_id': 'int64'}  # тип данных для колонки product_id
                                   #  nrows=100 (прочитать только первые 100 строк)
)

In [5]:
# Сортируем от новых к старым
df_sorted = df.sort_values('sale_date', ascending=False)

# Проверяем результат
display(df_sorted.head(10))

,product_id,category,model_name,sale_date,quantity,revenue,stock_on_hand
49083,5,Clothing,Model_5,2026-03-07,3,97.50,47
49048,19,Clothing,Model_19,2026-03-06,4,438.24,80
49032,3,Clothing,Model_3,2026-03-06,5,533.70,110
49033,9,Clothing,Model_9,2026-03-06,2,66.80,103
49034,8,Electronics,Model_8,2026-03-06,3,106.35,64
49035,7,Clothing,Model_7,2026-03-06,5,302.25,109
49036,13,Clothing,Model_13,2026-03-06,4,216.56,25
49037,13,Clothing,Model_13,2026-03-06,5,270.70,25
49038,12,Electronics,Model_12,2026-03-06,5,287.30,83
49039,4,Electronics,Model_4,2026-03-06,6,213.54,50


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49084 entries, 0 to 49083
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   product_id     49084 non-null  int64         
 1   category       49084 non-null  object        
 2   model_name     49084 non-null  object        
 3   sale_date      49084 non-null  datetime64[ns]
 4   quantity       49084 non-null  int64         
 5   revenue        49084 non-null  float64       
 6   stock_on_hand  49084 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(2)
memory usage: 2.6+ MB


In [7]:
print(f"Данные: {len(df)} строк, {df.columns.tolist()}")

Данные: 49084 строк, ['product_id', 'category', 'model_name', 'sale_date', 'quantity', 'revenue', 'stock_on_hand']


In [8]:
# Вариант Б: Запрос к БД 
# from src.db.queries import run_query ⬆
# QUERY = "SELECT ... -- запрос"
# df = run_query(QUERY) 

query = '''
SELECT 
    p.product_id,
    p.category,
    p.model_name,
    p.cost_price,
    p.sale_price,
    s.sale_id,
    s.sale_date::DATE AS sale_date,          -- Приводим к дате для удобства
    s.sale_date::TIME AS sale_time,          -- Выделяем время отдельно
    s.quantity,
    (s.quantity * p.sale_price) AS revenue,  -- Выручка по строке
    s.customer_id,
    inv.stock_on_hand,
    inv.operation_type
FROM sales s
JOIN products p ON s.product_id = p.product_id
LEFT JOIN inventory_log inv 
    ON s.product_id = inv.product_id 
    AND s.sale_date::DATE = inv.date
ORDER BY s.sale_date;
'''
df_2 = run_query(query)

# Преобразуем дату в datetime
df_2['sale_date'] = pd.to_datetime(df_2['sale_date'])

# Сортируем от новых к старым
df_sorted = df_2.sort_values('sale_date', ascending=False)

# Проверяем результат
display(df_sorted.head(10))

2026-03-10 22:38:37.874 | INFO     | src.db.queries:run_query:28 - Query returned 49084 rows


,product_id,category,model_name,cost_price,sale_price,sale_id,sale_date,sale_time,quantity,revenue,customer_id,stock_on_hand,operation_type
49083,5,Clothing,Model_5,23.92,32.50,49084,2026-03-07,00:00:00,3,97.50,950,47,snapshot
49048,19,Clothing,Model_19,11.09,109.56,49049,2026-03-06,13:15:00,4,438.24,317,80,snapshot
49032,3,Clothing,Model_3,20.09,106.74,49033,2026-03-06,06:15:00,5,533.70,24,110,snapshot
49033,9,Clothing,Model_9,31.98,33.40,49034,2026-03-06,06:30:00,2,66.80,364,103,snapshot
49034,8,Electronics,Model_8,51.93,35.45,49035,2026-03-06,07:15:00,3,106.35,224,64,snapshot
49035,7,Clothing,Model_7,42.21,60.45,49036,2026-03-06,07:30:00,5,302.25,140,109,snapshot
49036,13,Clothing,Model_13,47.09,54.14,49037,2026-03-06,07:45:00,4,216.56,926,25,snapshot
49037,13,Clothing,Model_13,47.09,54.14,49038,2026-03-06,08:30:00,5,270.70,896,25,snapshot
49038,12,Electronics,Model_12,45.72,57.46,49039,2026-03-06,10:30:00,5,287.30,735,83,snapshot
49039,4,Electronics,Model_4,18.08,35.59,49040,2026-03-06,10:45:00,6,213.54,419,50,snapshot


In [5]:
df_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49084 entries, 0 to 49083
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   product_id      49084 non-null  int64         
 1   category        49084 non-null  object        
 2   model_name      49084 non-null  object        
 3   cost_price      49084 non-null  float64       
 4   sale_price      49084 non-null  float64       
 5   sale_id         49084 non-null  int64         
 6   sale_date       49084 non-null  datetime64[ns]
 7   sale_time       49084 non-null  object        
 8   quantity        49084 non-null  int64         
 9   revenue         49084 non-null  float64       
 10  customer_id     49084 non-null  int64         
 11  stock_on_hand   49084 non-null  int64         
 12  operation_type  49084 non-null  object        
dtypes: datetime64[ns](1), float64(3), int64(5), object(4)
memory usage: 4.9+ MB


In [ ]:
# ЭКСПОРТ РЕЗУЛЬТАТОВ

# Сохранить очищенные данные
EXPORT_PATH = PROJECT_ROOT / "data" / "exports" / "00002_analysis_ready.csv"
EXPORT_PATH.parent.mkdir(exist_ok=True)
df.to_csv(EXPORT_PATH, index=False)
print(f"Экспорт: {EXPORT_PATH}")